# Imbalanced data

In machine learning, an imbalanced dataset is one in which some classes appear much more often than others. In binary classification, this usually means that one class dominates the dataset while the other is relatively rare.

Typical examples are fraud detection, disease diagnosis, and failure prediction. In all of them, the minority class is usually the one we care about the most.

This notebook studies that situation using a real binary classification dataset. The main objective is not only to train a classifier, but to understand why imbalance changes the way we evaluate models and choose mitigation strategies.

**Credits** - This notebook is adapted from material originally presented by Damini Dasgupta.

## Connection with the lecture notes

The lecture notes emphasize a key point: class imbalance is not just a data-distribution issue, but also an **evaluation** issue.

If the positive class is rare, a model can achieve high accuracy simply by predicting the majority class most of the time. That is why, throughout this notebook, we will pay special attention to:

* precision,
* recall,
* F1-score,
* confusion matrices.

These metrics help us measure what really matters: how well the model detects minority-class examples.

## Importing packages

In [ ]:
# pip install xgboost

import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

%matplotlib inline

warnings.filterwarnings("ignore")
pd.options.display.max_colwidth = 150


# Basic Exploratory Data Analysis

We start by loading the dataset and looking at its basic structure. The immediate goals are:

* confirm the target column,
* identify numerical and categorical variables,
* inspect whether the data appears to be clean enough for a first modeling pass,
* verify the class distribution of the target.

In [ ]:
data = pd.read_csv('../data/aug_train.csv')

print('Shape:', data.shape)
display(data.head(10))
display(data.describe())
data.info()

data[['Age', 'Annual_Premium', 'Policy_Sales_Channel', 'Vintage']].hist(figsize=(10, 6))
plt.tight_layout()
plt.show()

print('Missing values:', data.isnull().sum().sum())
print('\nColumn types:')
display(data.dtypes)

print('Gender values:', data.Gender.unique())
print('Vehicle_Age values:', data.Vehicle_Age.unique())
print('Vehicle_Damage values:', data.Vehicle_Damage.unique())


## Why imbalance matters here

The target variable is `Response`. Before training any model, we should quantify its imbalance.

This is an important habit: in imbalanced classification, the target distribution often tells us in advance that accuracy alone will be a weak metric.

In [ ]:
class_division = [data[data['Response'] == 1].shape[0], data[data['Response'] == 0].shape[0]]
my_labels = ['Positive Class', 'Negative Class']

plt.figure(figsize=(5, 5))
plt.pie(class_division, labels=my_labels, shadow=True, autopct='%1.1f%%')
plt.title('Class distribution of Response')
plt.show()

print('Proportion of Positive Class:', round(data[data['Response'] == 1].shape[0] / data.shape[0] * 100, 2), '%')


The positive class is clearly the minority class. This has two immediate consequences that match the lecture notes:

* the classifier may learn to favor the majority class;
* the default threshold of 0.5 may not be appropriate if recall on the minority class is important.

We will therefore compare several mitigation strategies instead of treating the baseline model as the final answer.

# Basic preprocessing

For this first version of the notebook, we will use one-hot encoding for the categorical variables and then build the feature matrix `X` and target vector `y`.

In [ ]:
data = pd.get_dummies(data, ['Gender', 'Vehicle_Age', 'Vehicle_Damage'], drop_first=True)

X = data.drop(columns=['id', 'Response'])
X.columns = [
    'Age', 'Driving_License', 'Region_Code', 'Previously_Insured',
    'Annual_Premium', 'Policy_Sales_Channel', 'Vintage',
    'Gender_Male', 'Vehicle_Age_lessthan_1_Year',
    'Vehicle_Age_greaterthan_2_Years', 'Vehicle_Damage_Yes'
]
y = data['Response']

display(X.head())
print('Feature matrix shape:', X.shape)
print('Target mean:', y.mean())


# Train/validation/test split

This notebook uses three subsets:

* training set: used to fit the model;
* validation set: used to choose decision thresholds;
* test set: used only for the final comparison of trained strategies.

The split is stratified so that the class proportion is preserved in all subsets. This is especially important in imbalanced classification, otherwise one split could become much easier or much harder than the others.

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.3, random_state=42, stratify=y_temp
)

print('Proportion of positive class in original set:', round(y.sum() / len(y) * 100, 2), '%')
print('Proportion of positive class in train set:   ', round(y_train.sum() / len(y_train) * 100, 2), '%')
print('Proportion of positive class in val set:     ', round(y_val.sum() / len(y_val) * 100, 2), '%')
print('Proportion of positive class in test set:    ', round(y_test.sum() / len(y_test) * 100, 2), '%')


# Baseline model

We first train a standard `XGBClassifier` with default settings. This baseline is useful because it shows what happens when we ignore the imbalance problem and simply use the default classification rule.

The baseline helps answer two questions:

* how misleading is accuracy in this problem?
* how much recall are we losing on the minority class?

In [ ]:
xgb_classifier = xgb.XGBClassifier()
xgb_classifier.fit(X_train, y_train)

y_pred_basic = xgb_classifier.predict(X_test)

print('Accuracy of test set:', round(accuracy_score(y_test, y_pred_basic) * 100, 2), '%')


In [ ]:
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix'):
    conf_matrix = confusion_matrix(y_true=y_true, y_pred=y_pred)

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.matshow(conf_matrix, cmap=plt.cm.Oranges, alpha=0.3)
    for i in range(conf_matrix.shape[0]):
        for j in range(conf_matrix.shape[1]):
            ax.text(x=j, y=i, s=conf_matrix[i, j], va='center', ha='center', size='xx-large')

    plt.xlabel('Predictions', fontsize=14)
    plt.ylabel('Actuals', fontsize=14)
    plt.title(title, fontsize=14)
    plt.show()


def compute_performance_metrics(y_true, y_pred):
    precision = round(precision_score(y_true, y_pred), 4)
    recall = round(recall_score(y_true, y_pred), 4)
    f1 = round(f1_score(y_true, y_pred), 4)
    return [precision, recall, f1]


plot_confusion_matrix(y_test, y_pred_basic, title='Baseline model')

metrics_basic = compute_performance_metrics(y_test, y_pred_basic)
original = pd.DataFrame(metrics_basic, index=['Precision', 'Recall', 'F1_score'], columns=['Basic Model'])
original['Comments'] = [
    'A substantial fraction of predicted positives are false positives.',
    'Many truly positive cases are still being missed.',
    'The baseline trade-off is not necessarily aligned with the application goal.'
]
display(original)


The baseline already illustrates the central lesson from the notes: a reasonable accuracy score does not guarantee good minority-class detection.

From here on, we will compare four common strategies for dealing with class imbalance:

1. adjusting the decision threshold,
2. random oversampling,
3. random undersampling,
4. synthetic oversampling with SMOTE,
5. class weighting in the learning algorithm.

Each strategy changes the precision-recall trade-off in a different way.

# Strategy 1: adjusting the classification threshold

Models such as XGBoost output probabilities. By default, the positive class is predicted when the estimated probability is at least `0.5`.

In imbalanced problems, that default threshold is often too conservative for the minority class. Lowering the threshold usually increases recall, but it may reduce precision.

The important methodological detail is this: we choose the threshold on the **validation set**, and only then evaluate the chosen threshold on the **test set**.

In [ ]:
cols = ['Metrics', 'Original']
adj_threshold = pd.DataFrame()
adj_threshold['Original'] = metrics_basic
thresholds = [j / 20 for j in range(1, 16)]

for threshold in thresholds:
    y_pred_new_threshold = (xgb_classifier.predict_proba(X_val)[:, 1] >= threshold).astype(int)

    precision_adj_threshold = round(precision_score(y_val, y_pred_new_threshold), 4)
    recall_adj_threshold = round(recall_score(y_val, y_pred_new_threshold), 4)
    f1_score_adj_threshold = round(f1_score(y_val, y_pred_new_threshold), 4)
    metrics_adj_threshold = [precision_adj_threshold, recall_adj_threshold, f1_score_adj_threshold]
    name = 'Threshold: ' + str(threshold)
    adj_threshold[name] = metrics_adj_threshold
    cols.append(name)

adj_threshold.index = ['Precision', 'Recall', 'F1_score']
adj_threshold = adj_threshold.reset_index()
adj_threshold.columns = cols

display(adj_threshold.transpose())

adj_threshold_plot = adj_threshold.copy()
adj_threshold_plot.columns = ['Metrics'] + list(adj_threshold.columns[1:])
adj_threshold_plot.plot(
    x='Metrics', kind='bar', stacked=False, title='Metrics across varying thresholds',
    figsize=(15, 5), cmap='twilight'
).legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.show()


From the validation results, we choose a threshold of `0.3`. The exact choice is not universal; it depends on the application and on which trade-off we want to prioritize.

Here, the threshold is chosen because it substantially improves recall and F1 compared with the baseline, while keeping precision at a still usable level.

In [ ]:
optimal_threshold = 0.3
y_pred_new_threshold = (xgb_classifier.predict_proba(X_test)[:, 1] >= optimal_threshold).astype(int)

plot_confusion_matrix(y_test, y_pred_new_threshold, title='Threshold-adjusted model')

metrics_changedthreshold = compute_performance_metrics(y_test, y_pred_new_threshold)

changedthreshold = pd.DataFrame(
    list(zip(metrics_basic, metrics_changedthreshold)),
    index=['Precision', 'Recall', 'F1_score'],
    columns=['Basic Model', 'Threshold = 0.3']
)
display(changedthreshold)


# Strategy 2: random oversampling

Random oversampling duplicates minority-class examples in the training set until the classes become balanced.

The key idea is simple: if the model sees the minority class more often during training, it may learn patterns that were previously overshadowed by the majority class.

Important methodological point: resampling must be applied **only to the training set**. The validation and test sets should preserve the original distribution.

In [ ]:
train_data = pd.concat([X_train, y_train], axis=1)

Response_Zero = train_data[train_data.Response == 0]
Response_One = train_data[train_data.Response == 1]

upsampled_One = resample(
    Response_One,
    replace=True,
    n_samples=len(Response_Zero),
    random_state=27
)

upsampled = pd.concat([Response_Zero, upsampled_One])

print('Before oversampling:')
display(y_train.value_counts())
print('After oversampling:')
display(upsampled['Response'].value_counts())

X_train_upsampled = upsampled.drop(columns=['Response'])
y_train_upsampled = upsampled['Response']

xgb_upsampled = xgb.XGBClassifier()
xgb_upsampled.fit(X_train_upsampled, y_train_upsampled)

y_pred_upsampled = xgb_upsampled.predict(X_test)

plot_confusion_matrix(y_test, y_pred_upsampled, title='Random oversampling')

metrics_upsampled = compute_performance_metrics(y_test, y_pred_upsampled)

upsampled_results = pd.DataFrame(
    list(zip(metrics_basic, metrics_changedthreshold, metrics_upsampled)),
    index=['Precision', 'Recall', 'F1_score'],
    columns=['Basic Model', 'Threshold = 0.3', 'Upsampled Dataset']
)
display(upsampled_results)


# Strategy 3: random undersampling

Random undersampling removes examples from the majority class until the dataset becomes balanced.

This often increases recall, because the classifier is no longer dominated by the majority class during training. The downside is that we throw away many training examples, which may remove useful information.

In [ ]:
downsampled_Zero = resample(
    Response_Zero,
    replace=False,
    n_samples=len(Response_One),
    random_state=27
)

downsampled = pd.concat([downsampled_Zero, Response_One])

print('Before undersampling:')
display(y_train.value_counts())
print('After undersampling:')
display(downsampled['Response'].value_counts())

X_train_downsampled = downsampled.drop(columns=['Response'])
y_train_downsampled = downsampled['Response']

xgb_downsampled = xgb.XGBClassifier()
xgb_downsampled.fit(X_train_downsampled, y_train_downsampled)

y_pred_downsampled = xgb_downsampled.predict(X_test)

plot_confusion_matrix(y_test, y_pred_downsampled, title='Random undersampling')

metrics_downsampled = compute_performance_metrics(y_test, y_pred_downsampled)

downsampled_results = pd.DataFrame(
    list(zip(metrics_basic, metrics_changedthreshold, metrics_upsampled, metrics_downsampled)),
    index=['Precision', 'Recall', 'F1_score'],
    columns=['Basic Model', 'Threshold = 0.3', 'Upsampled Dataset', 'Downsampled Dataset']
)
display(downsampled_results)


# Strategy 4: synthetic oversampling with SMOTE

The `imbalanced-learn` library provides several tools specifically designed for imbalanced datasets. One of the most popular is **SMOTE**.

Instead of duplicating minority examples, SMOTE creates **synthetic** minority instances by interpolating between nearby minority points.

That is why SMOTE is often described as a compromise between simple oversampling and a more informed data-generation strategy.

For a minority example $x_i$ and one of its minority-class nearest neighbors $x_{zi}$, SMOTE generates a synthetic point using:

$$
x_{new} = x_i + \lambda (x_{zi} - x_i)
$$

where $\lambda \in [0, 1]$. This places the new point somewhere on the line segment between the two examples.

<p align="center">
<img src="https://imbalanced-learn.org/stable/_images/sphx_glr_plot_illustration_generation_sample_001.png" alt="SMOTE illustration" width="400"/>
</p>


In [ ]:
# pip install -U imbalanced-learn

from imblearn.over_sampling import SMOTE

smote_oversample = SMOTE()
X_train_smote, y_train_smote = smote_oversample.fit_resample(X_train, y_train)

print('Class distribution after SMOTE:')
print(Counter(y_train_smote))

xgb_smote = xgb.XGBClassifier()
xgb_smote.fit(X_train_smote, y_train_smote)

y_pred_smote = xgb_smote.predict(X_test)

plot_confusion_matrix(y_test, y_pred_smote, title='SMOTE')

metrics_smote = compute_performance_metrics(y_test, y_pred_smote)

results = pd.DataFrame(
    list(zip(metrics_basic, metrics_changedthreshold, metrics_upsampled, metrics_downsampled, metrics_smote)),
    index=['Precision', 'Recall', 'F1_score'],
    columns=['Basic Model', 'Threshold = 0.3', 'Upsampled Dataset', 'Downsampled Dataset', 'SMOTE upsampled']
)
display(results)


# Strategy 5: class weighting

A different family of solutions changes not the data, but the **loss function** seen by the learning algorithm.

In XGBoost, the parameter `scale_pos_weight` increases the importance of the positive class during training. This is often attractive because it avoids explicit resampling while still forcing the learner to pay more attention to the minority class.

A common heuristic is:

$$
\texttt{scale\_pos\_weight} = \frac{\#\text{negative}}{\#\text{positive}}
$$

which is exactly what we use below.

In [ ]:
xgb_weighted = xgb.XGBClassifier(
    scale_pos_weight=len(y_train[y_train == 0]) / len(y_train[y_train == 1])
)

xgb_weighted.fit(X_train, y_train)

y_pred_weighted = xgb_weighted.predict(X_test)

plot_confusion_matrix(y_test, y_pred_weighted, title='Weighted model')

metrics_weighted = compute_performance_metrics(y_test, y_pred_weighted)

results = pd.DataFrame(
    list(zip(metrics_basic, metrics_changedthreshold, metrics_upsampled, metrics_downsampled, metrics_smote, metrics_weighted)),
    index=['Precision', 'Recall', 'F1_score'],
    columns=['Basic Model', 'Threshold = 0.3', 'Upsampled Dataset', 'Downsampled Dataset', 'SMOTE upsampled', 'Weighted']
)
display(results)


# Final discussion

At this stage, the notebook allows us to compare several families of solutions side by side.

The main didactic takeaway is not that one technique is universally best. The real lesson is that imbalance changes the optimization and evaluation landscape:

* threshold adjustment changes the decision rule without retraining;
* random oversampling and SMOTE change the training distribution;
* random undersampling trades information for balance;
* class weighting changes the loss seen by the learner.

In practice, the best choice depends on the cost of false positives versus false negatives, the model family, and the amount of data available.

## Suggested exercises

To connect this notebook more directly with the lecture notes, useful follow-up exercises are:

1. add ROC and precision-recall curves for the baseline and weighted models;
2. choose the threshold by maximizing recall subject to a minimum precision constraint;
3. compare threshold adjustment and class weighting under the same business objective;
4. place SMOTE inside a cross-validation pipeline and discuss why doing it before the split would cause leakage;
5. tune `scale_pos_weight` instead of using only the default heuristic.

These extensions turn the notebook from a catalog of techniques into a more principled study of imbalanced classification.